# Lab 2 · Python thuần trên dữ liệu thật

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành · bài 2**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Trong giờ lý thuyết, bạn đã xử lý một bảng 8 phòng tự tạo. Trong buổi lab này, bạn xử lý
**18.534 phòng thật** của Santiago (Chile)
chỉ bằng Python thuần, chưa cần pandas.

## Cách làm việc trong buổi lab

- Bài tập được chia thành từng bước; mỗi bước có ô `TODO` và phần kiểm tra `assert`.
  Hoàn thành toàn bộ `assert` nghĩa là kết quả đáp ứng yêu cầu.
- Phần khởi động và bài có hướng dẫn: bạn nên **tự gõ, không dùng AI** — các bài kiểm tra
  định kỳ 🚫 đóng ở giờ lý thuyết đánh giá các kỹ năng này.
- Bài tự làm ở cuối được gắn nhãn ✅ mở: bạn được dùng AI, kèm trách nhiệm khai báo
  theo chính sách AI của môn.
- Nếu chưa giải quyết được một bước sau 3 phút, hãy gọi giảng viên thực hành đến hỗ trợ.

## Mục tiêu

Sau buổi lab, bạn sẽ:

1. Đọc được file CSV thật bằng `csv.DictReader` và biết mọi giá trị đọc lên đều là chuỗi.
2. Viết được hàm làm sạch chịu được dữ liệu bẩn, bắt đúng loại lỗi bằng `try/except`.
3. Tổng hợp được dữ liệu theo nhóm bằng comprehension, dict và set.
4. Ghi kết quả ra file JSON để chương trình khác có thể tiếp tục xử lý.

## Phần 0 · Khởi động (~10 phút)

Ba bài nhỏ giúp bạn củng cố các thao tác cốt lõi của bài giảng.

In [2]:
# W1 — f-string: hai đuôi định dạng hay dùng nhất
gia_tb = 118199.6      # CLP (peso Chile)
ty_le = 0.178

# TODO: tạo chuỗi ket_qua có dạng 'Giá TB: 118,200 CLP' (dùng đuôi :,.0f)
ket_qua = f"Giá TB: {gia_tb:,.0f} CLP"
# TODO: tạo chuỗi ty_le_chu có dạng '17.8%' (dùng đuôi :.1%)
ty_le_chu = f"{ty_le:.1%}"

# --- Ô kiểm tra: không sửa phần dưới ---
assert ket_qua == "Giá TB: 118,200 CLP", f"đang ra: {ket_qua!r}"
assert ty_le_chu == "17.8%", f"đang ra: {ty_le_chu!r}"
print("W1 ổn:", ket_qua, "|", ty_le_chu)

W1 ổn: Giá TB: 118,200 CLP | 17.8%


In [3]:
# W2 — None là "thiếu", không phải 0
gia = [52000, None, 78000, None, 61000]

# TODO: dùng comprehension lấy các giá khác None, rồi tính trung bình của chúng
gia_co = [x for x in gia if x != None]
tb = sum(gia_co) / len(gia_co)

# --- Ô kiểm tra ---
assert len(gia_co) == 3 and round(tb) == 63667, f"gia_co={gia_co}, tb={tb}"
print(f"W2 ổn: trung bình {tb:,.0f} CLP trên {len(gia_co)} phòng có giá")

W2 ổn: trung bình 63,667 CLP trên 3 phòng có giá


In [4]:
# W3 — set: so sánh hai snapshot
thang_3 = {101, 102, 103, 104}
thang_6 = {102, 104, 105}

# TODO: phòng có ở tháng 3 nhưng biến mất ở tháng 6; phòng mới xuất hiện ở tháng 6
bien_mat = thang_3 - thang_6
moi_them = thang_6 - thang_3

# --- Ô kiểm tra ---
assert bien_mat == {101, 103} and moi_them == {105}
print("W3 ổn — biến mất:", bien_mat, "· mới thêm:", moi_them)

W3 ổn — biến mất: {101, 103} · mới thêm: {105}


## Phần 1 · Khảo sát thị trường Airbnb Santiago (~60 phút)

Tình huống: bạn là phân tích viên của một công ty du lịch. Công ty cần một bản tổng quan
về thị trường Airbnb Santiago — **giá cả thế nào, khu nào đắt, dữ liệu có đáng tin không** —
và muốn nhận kết quả dưới dạng file JSON để đưa vào hệ thống khác.

Dữ liệu: bảng `listings` bản rút gọn (thư mục *visualisations*) của Inside Airbnb,
snapshot ngày 29/06/2026. Giá tính bằng **CLP**
(peso Chile) cho một đêm.

### Bước 1 · Tải và đọc dữ liệu

Ô dưới đã viết sẵn — bạn chạy và đọc kỹ từng dòng trước khi đi tiếp.

In [5]:
import csv
import urllib.request

URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/visualisations/listings.csv")
urllib.request.urlretrieve(URL, "santiago.csv")

with open("santiago.csv", encoding="utf-8") as f:
    ds = list(csv.DictReader(f))     # list các dict — mỗi dict là một phòng

print("Số phòng:", len(ds))
ds[0]

Số phòng: 18534


{'id': '978070332077815549',
 'name': 'luminosa mansarda con balcón',
 'host_id': '118157228',
 'host_profile_id': '1468207531264648757',
 'host_name': 'Patricia',
 'neighbourhood_group': '',
 'neighbourhood': 'Ñuñoa',
 'latitude': '-33.43765',
 'longitude': '-70.5833',
 'room_type': 'Private room',
 'price': '45647',
 'minimum_nights': '4',
 'number_of_reviews': '2',
 'last_review': '2023-12-15',
 'reviews_per_month': '0.06',
 'calculated_host_listings_count': '5',
 'availability_365': '269',
 'number_of_reviews_ltm': '0',
 'license': ''}

In [6]:
# --- Ô kiểm tra ---
assert len(ds) == 18534
print("Đọc đủ 18.534 phòng.")

Đọc đủ 18.534 phòng.


Nhìn kỹ dòng vừa in: **mọi giá trị đều là chuỗi**, kể cả `price` và `minimum_nights` —
CSV không mang kiểu dữ liệu, người đọc phải tự ép kiểu.

Các cột sẽ dùng trong lab:

| Cột | Nghĩa |
|---|---|
| `neighbourhood` | khu (đơn vị hành chính của Santiago) |
| `room_type` | loại phòng (Entire home/apt, Private room…) |
| `price` | giá một đêm, CLP — **có ô bỏ trống** |
| `number_of_reviews` | tổng số review của phòng |
| `last_review` | ngày có review gần nhất — trống nếu chưa từng có |

### Bước 2 · Hàm làm sạch giá

`float("45647")` trả kết quả, nhưng `float("")` phát sinh `ValueError`, còn `float(None)` phát sinh
`TypeError`. Hàm làm sạch cần xử lý được cả ba trường hợp.

In [7]:
def to_float(s):
    """Đổi chuỗi số sang float; chuỗi rỗng / hỏng / None thì trả về None."""
    # TODO: viết thân hàm — try/except, bắt đúng hai loại lỗi nêu trên
    try:
      return float(s)
    except:
      return None

# --- Ô kiểm tra ---
assert to_float("45647") == 45647.0
assert to_float("59000.5") == 59000.5
assert to_float("") is None
assert to_float("N/A") is None
assert to_float(None) is None
print("to_float qua cả 5 ca thử.")

to_float qua cả 5 ca thử.


### Bước 3 · Áp hàm cho cả cột giá

In [8]:
# TODO: dùng comprehension áp to_float cho price của MỌI phòng trong ds
gia = [to_float(x["price"]) for x in ds]
# TODO: lọc lấy các giá khác None (chú ý: dùng `is not None`)
gia_hop_le = [x for x in gia if x is not None]
# TODO: số phòng không khai giá
so_thieu = len(gia) - len(gia_hop_le)

# --- Ô kiểm tra ---
assert len(gia) == 18534 and len(gia_hop_le) == 17688 and so_thieu == 846
print(f"{so_thieu} phòng không khai giá ({so_thieu / len(ds):.1%}).")

846 phòng không khai giá (4.6%).


Câu hỏi: vì sao đề yêu cầu `if g is not None` chứ không phải `if g`?
Vì `if g` loại luôn cả giá `0.0`. Snapshot này không có phòng giá 0, nhưng quy tắc lọc
phải là **quyết định chủ động** của bạn — không phải hiệu ứng phụ của code.

### Bước 4 · So sánh trung bình và trung vị

Hàm trung vị đã viết sẵn (sắp xếp rồi lấy phần tử giữa):

In [9]:
def trung_vi(xs):
    """Trung vị của một list số (không rỗng)."""
    xs = sorted(xs)
    n = len(xs)
    if n % 2 == 1:
        return xs[n // 2]
    return (xs[n // 2 - 1] + xs[n // 2]) / 2

assert trung_vi([1, 9, 5]) == 5 and trung_vi([1, 2, 3, 10]) == 2.5
print("trung_vi sẵn sàng.")

trung_vi sẵn sàng.


In [10]:
# TODO: tính giá trung bình (sum / len) và giá trung vị của gia_hop_le
tb = sum(gia_hop_le) / len(gia_hop_le)
tv = trung_vi(gia_hop_le)

# --- Ô kiểm tra ---
assert round(tb) == 118200 and tv == 59000.0
print(f"Trung bình: {tb:,.0f} CLP — Trung vị: {tv:,.0f} CLP")

Trung bình: 118,200 CLP — Trung vị: 59,000 CLP


Trung bình **gấp đôi** trung vị, cho thấy một số giá trị rất lớn đang kéo trung bình lên.
Hãy tìm phòng có giá lớn nhất:

In [11]:
co_gia = [r for r in ds if to_float(r["price"]) is not None]

# TODO: dùng max với key=lambda để tìm phòng có giá cao nhất trong co_gia
dat_nhat = max(co_gia, key = lambda x : to_float(x["price"]))
# --- Ô kiểm tra ---
assert to_float(dat_nhat["price"]) == 97000045.0
print(dat_nhat["name"], "—", dat_nhat["neighbourhood"])
print(f"Giá: {to_float(dat_nhat['price']):,.0f} CLP/đêm")

Apartasuites EL CALEÑO 2 con estacionamiento priva — Santiago
Giá: 97,000,045 CLP/đêm


97 triệu CLP (khoảng 2,6 tỷ đồng) cho một đêm. Giá thật hay lỗi nhập liệu? Chưa thể biết —
nhưng một mình phòng này đủ kéo trung bình toàn thành phố lên thêm ~5.500 CLP.
Bài 10 sẽ trình bày cách **gắn cờ** các giá trị cần kiểm tra. Với phân phối lệch trong ví dụ này,
trung vị phù hợp hơn trung bình để mô tả mức giá điển hình.

### Bước 5 · Đếm phòng theo khu

Mẫu "đếm theo nhóm" bằng dict tương ứng với thao tác `groupby` của pandas
được học ở bài 5:

In [12]:
dem_khu = {}
for r in ds:
    khu = r["neighbourhood"]
    # TODO: cộng dồn 1 vào dem_khu[khu] (gợi ý: dem_khu.get(khu, 0))
    dem_khu[khu] = dem_khu.get(khu,0) + 1

# --- Ô kiểm tra ---
assert len(dem_khu) == 31 and dem_khu["Santiago"] == 7182 and dem_khu["Ñuñoa"] == 1813
print(f"{len(dem_khu)} khu; riêng khu trung tâm (trùng tên Santiago) có {dem_khu['Santiago']:,} phòng.")

31 khu; riêng khu trung tâm (trùng tên Santiago) có 7,182 phòng.


In [13]:
# TODO: sắp xếp dem_khu.items() theo số phòng giảm dần, lấy 5 khu đông nhất
top5 = sorted(dem_khu.items(),key = lambda x: to_float(x[1]),reverse=True)[:5]

# --- Ô kiểm tra ---
assert top5[0] == ("Santiago", 7182) and len(top5) == 5
for ten, n in top5:
    print(f"{ten:<15}{n:>7,}")

Santiago         7,182
Providencia      2,997
Las Condes       2,862
Ñuñoa            1,813
Lo Barnechea       824


### Bước 6 · So sánh giá giữa các khu

So giá **trung vị** giữa 5 khu đông phòng nhất. Viết hàm phụ trước — mỗi việc một hàm:

In [14]:
def gia_khu(ten):
    """List giá hợp lệ (float) của một khu."""
    # TODO: một comprehension — lọc theo khu, ép kiểu bằng to_float, bỏ None
    return [to_float(p["price"]) for p in co_gia if p["neighbourhood"] == ten]

# TODO: dict comprehension {tên khu: trung vị giá của khu} cho 5 khu trong top5
tv_khu = {x[0] : trung_vi(gia_khu(x[0])) for x in top5}

# --- Ô kiểm tra ---
assert tv_khu["Santiago"] == 47755.0 and tv_khu["Lo Barnechea"] == 426230.0
for ten, m in sorted(tv_khu.items(), key=lambda kv: kv[1], reverse=True):
    print(f"{ten:<15}{m:>12,.0f} CLP")

Lo Barnechea        426,230 CLP
Las Condes           97,460 CLP
Providencia          72,845 CLP
Ñuñoa                60,981 CLP
Santiago             47,755 CLP


Trung vị giá của Lo Barnechea cao gấp khoảng 9 lần khu trung tâm. Kết quả này cho thấy
chênh lệch không chỉ do một vài phòng ngoại lai. Với phân phối giá lệch, trung vị ít nhạy
với các giá trị cực lớn hơn trung bình.

### Bước 7 · Phòng chưa từng có review

Bao nhiêu phần thị trường chưa từng được khách đánh giá? Cẩn thận:
`number_of_reviews` là **chuỗi**, nên phải so sánh với `"0"`, không phải `0`.

In [15]:
# TODO: lọc các phòng có number_of_reviews == "0"
chua_review = [x for x in ds if x["number_of_reviews"] == "0"]

# --- Ô kiểm tra ---
assert len(chua_review) == 3291
print(f"{len(chua_review):,} phòng chưa có review ({len(chua_review) / len(ds):.1%}).")

# Kiểm tra chéo: phòng chưa review thì last_review phải trống — dữ liệu có nhất quán không?
assert all(r["last_review"] == "" for r in chua_review)
print("Nhất quán: mọi phòng chưa review đều có last_review trống.")

3,291 phòng chưa có review (17.8%).
Nhất quán: mọi phòng chưa review đều có last_review trống.


Thao tác vừa làm là một quy tắc QA: **kiểm tra chéo hai cột** mô tả cùng một hiện tượng.
Đến bài 10, các quy tắc như vậy sẽ được tổ chức thành một bộ kiểm tra hoàn chỉnh.

### Bước 8 · Xuất báo cáo JSON

Tập hợp các số liệu vào một dict và ghi ra file JSON để chương trình khác có thể đọc:

In [16]:
import json

bao_cao = {
    "thanh_pho": "Santiago",
    "snapshot": "2026-06-29",
    "so_phong": len(ds),
    "so_phong_co_gia": len(gia_hop_le),
    "gia_trung_vi_clp": tv,
    "ty_le_chua_review": round(len(chua_review) / len(ds), 3),
    "gia_trung_vi_theo_khu": tv_khu,
}
with open("bao_cao_santiago.json", "w", encoding="utf-8") as f:
    json.dump(bao_cao, f, ensure_ascii=False, indent=2)

# TODO: đọc lại file vừa ghi (json.load) vào biến doc_lai
with open("bao_cao_santiago.json","r", encoding="utf-8") as f:
  doc_lai = json.load(f)

# --- Ô kiểm tra ---
assert doc_lai == bao_cao
print("Ghi và đọc lại khớp nhau. Mở tab Files (📁 bên trái) để xem file.")

Ghi và đọc lại khớp nhau. Mở tab Files (📁 bên trái) để xem file.


## Phần 2 · Bài tự làm ✅ mở (làm sớm tại lớp hoặc làm tại nhà)

Hai bài dưới đây bạn **được dùng AI**, theo quy trình 5 bước của bài 1. Nếu dùng, ghi lại
ngay trong notebook: **prompt chính bạn đã hỏi, và bạn đã kiểm chứng kết quả bằng cách nào** —
thực hành cách khai báo sẽ được dùng trong file `AI_USAGE.md`.

### Tự làm 1 · Bảng xếp hạng khu

In bảng xếp hạng **mọi khu có ≥ 100 phòng có giá hợp lệ** (không chỉ top 5): tên khu,
số phòng có giá, giá trung vị — xếp theo trung vị giảm dần, cột căn thẳng bằng f-string.

Kết quả đúng có **13 khu**; Lo Barnechea đứng đầu, Estación Central cuối bảng.

In [17]:
# TODO Tự làm 1 — viết tự do
# (gợi ý: tái dùng gia_khu + trung_vi; định dạng cột: f"{ten:<20}{n:>10,}{m:>14,.0f}")

co_gia = [r for r in ds if to_float(r["price"]) is not None]

dem_khu = {}
for r in co_gia:
  khu = r["neighbourhood"]
  dem_khu[khu] = dem_khu.get(khu,0) + 1

bang_xep_hang = [(x[0],x[1],trung_vi(gia_khu(x[0]))) for x in dem_khu.items() if x[1] >= 100]

bang_xep_hang.sort(key=lambda x: x[2],reverse=True)
for ten,n,m in bang_xep_hang:
  print(f"{ten:<20}{n:>10,}{m:>14,.0f}")

Lo Barnechea               791       426,230
Vitacura                   328       129,213
Las Condes               2,742        97,460
Providencia              2,851        72,845
Ñuñoa                    1,723        60,981
La Florida                 353        52,307
La Reina                   138        47,944
San Miguel                 178        47,926
Santiago                 6,884        47,755
Recoleta                   257        47,304
Independencia              152        46,395
Macul                      193        45,191
Estación Central           429        39,941


### Tự làm 2 · Dòng thời gian review (nâng cao)

Bảng thứ hai của Inside Airbnb: `reviews.csv` — **690.112 dòng**, chỉ 2 cột
(`listing_id`, `date`). Hãy:

1. Tải về từ URL cho sẵn rồi đọc bằng `csv.DictReader`.
2. Đếm số review theo **năm** (lát cắt chuỗi `date[:4]` + dict cộng dồn).
3. In mỗi năm một dòng, rồi trả lời: năm nào nhiều review nhất? Mức giảm năm 2020–2021
   do đâu? Số của 2026 thấp hơn 2025 — có kết luận được "thị trường đang co lại" không,
   khi snapshot được chụp ngày 29/06/2026?

In [18]:
URL_REVIEWS = ("https://data.insideairbnb.com/chile/rm/santiago/"
               "2026-06-29/visualisations/reviews.csv")
urllib.request.urlretrieve(URL_REVIEWS, "reviews.csv")

with open("reviews.csv","r",encoding = "utf-8") as f:
  ds = list(csv.DictReader(f))
ds[:6]

[{'listing_id': '52811', 'date': '2010-11-13'},
 {'listing_id': '52811', 'date': '2011-03-23'},
 {'listing_id': '52811', 'date': '2012-01-23'},
 {'listing_id': '52811', 'date': '2012-04-01'},
 {'listing_id': '52811', 'date': '2012-08-21'},
 {'listing_id': '52811', 'date': '2013-01-10'}]

## Tóm tắt buổi lab

| Nội dung chính | Sẽ gặp lại ở |
|---|---|
| Đọc CSV thật — mọi giá trị là chuỗi | pandas tự suy luận kiểu dữ liệu (bài 4) |
| Hàm làm sạch + try/except đúng loại lỗi | xây dựng các bước của pipeline |
| Trung bình vs trung vị; gắn cờ giá bất thường | làm sạch dữ liệu (bài 10) |
| Đếm / tổng hợp theo nhóm bằng dict | groupby (bài 5) |
| Xuất báo cáo JSON | API (bài 6) và LLM (bài 11) |

Buổi lý thuyết tiếp theo: **NumPy** — dùng phép toán vector hoá thay cho vòng lặp Python.